In [15]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit_aer import AerSimulator
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [40]:
# --- Helper Function ---
def get_quantum_random_bits(num_bits):
    """Generates a list of random bits by measuring a quantum superposition."""
    qc = QuantumCircuit(num_bits, num_bits)
    qc.h(range(num_bits))
    qc.measure(range(num_bits), range(num_bits))

    backend = AerSimulator()
    t_qc = transpile(qc, backend)
    result = backend.run(t_qc, shots=1).result()
    counts = result.get_counts()
    bitstring = list(counts.keys())[0]

    return [int(b) for b in bitstring[::-1]]

# --- Setup ---
N = 20
print(f"Generating {N} random bits and bases using Quantum Measurements...\n")

# === ALICE ===
alice_bits = get_quantum_random_bits(N)
alice_bases = get_quantum_random_bits(N)

# === BOB ===
bob_bases = get_quantum_random_bits(N)

# === EVE ===
eve_bases = get_quantum_random_bits(N)
eve_attack_choices = [1] * N

backend = AerSimulator()

# ---------------------------------------------------------
# PHASE 1: ALICE SENDS TO EVE
# ---------------------------------------------------------
qc_alice_eve = QuantumCircuit(N, N)

# Alice prepares her qubits
for i in range(N):
    if alice_bits[i] == 1:
        qc_alice_eve.x(i)
    if alice_bases[i] == 1:
        qc_alice_eve.h(i)

# Eve intercepts and measures only the ones she chose to attack
for i in range(N):
    if eve_attack_choices[i] == 1:
        if eve_bases[i] == 1:
            qc_alice_eve.h(i)
        qc_alice_eve.measure(i, i)

# Run Circuit 1 to force the collapse and get Eve's classical bits
t_qc_ae = transpile(qc_alice_eve, backend)
result_ae = backend.run(t_qc_ae, shots=1).result()
counts_ae = result_ae.get_counts()
eve_bitstring = list(counts_ae.keys())[0]

# Any qubit Eve didn't measure defaults to 0, which we filter out later
eve_raw_bits = [int(b) for b in eve_bitstring[::-1]]


# ---------------------------------------------------------
# PHASE 2: EVE (AND UNTOUCHED QUBITS) SENT TO BOB
# ---------------------------------------------------------
qc_eve_bob = QuantumCircuit(N, N)

for i in range(N):
    if eve_attack_choices[i] == 1:
        # 1. THE ATTACKED QUBITS: Eve re-prepares them based on her stolen measurements
        if eve_raw_bits[i] == 1:
            qc_eve_bob.x(i)
        if eve_bases[i] == 1:
            qc_eve_bob.h(i)
    else:
        # 2. THE UNTOUCHED QUBITS: They retain Alice's exact original quantum state
        if alice_bits[i] == 1:
            qc_eve_bob.x(i)
        if alice_bases[i] == 1:
            qc_eve_bob.h(i)

# Bob receives and measures all the qubits
for i in range(N):
    if bob_bases[i] == 1:
        qc_eve_bob.h(i)
    qc_eve_bob.measure(i, i)

# Run Circuit 2 to get Bob's classical bits
t_qc_eb = transpile(qc_eve_bob, backend)
result_eb = backend.run(t_qc_eb, shots=1).result()
counts_eb = result_eb.get_counts()
bob_bitstring = list(counts_eb.keys())[0]
bob_bits = [int(b) for b in bob_bitstring[::-1]]


# --- Post-Processing (Classical Communication) ---
# === ALICE & BOB ===
sifted_alice_key = []
sifted_bob_key = []

# 1. Key Sifting
for i in range(N):
    if alice_bases[i] == bob_bases[i]:
        sifted_alice_key.append(alice_bits[i])
        sifted_bob_key.append(bob_bits[i])

# 2. Error Checking (Sacrificing a subset)
sample_size = int(len(sifted_alice_key) *0.8)
alice_sample = sifted_alice_key[:sample_size]
bob_sample = sifted_bob_key[:sample_size]

# Clean up Eve's stolen bits for printing (ignoring the ones she chose not to measure)
eve_stolen_bits = [eve_raw_bits[i] if eve_attack_choices[i] == 1 else None for i in range(N)]

print("--- Attack Details ---")
print(f"Eve's Attack Pattern (1=Attacked): {eve_attack_choices}")
print(f"Eve's Stolen Bits (None=Ignored):  {eve_stolen_bits}")
attacks_made = sum(eve_attack_choices)
print(f"Eve attacked {attacks_made} out of {N} qubits ({(attacks_made/N)*100:.0f}%).")

print(f"\nSifted Key Length: {len(sifted_alice_key)}")
print(f"Sacrificed for testing: {sample_size} bits")

# Check for errors in the sacrificed subset
errors = sum(1 for a, b in zip(alice_sample, bob_sample) if a != b)
error_rate = (errors / sample_size) if sample_size > 0 else 0

print(f"\nError Rate in Sample: {error_rate * 100:.2f}%")

THRESHOLD = 0.10 # 10% tolerance threshold

if error_rate > THRESHOLD:
    print(f"Error rate exceeds {(THRESHOLD*100):.0f}%. An eavesdropper (Eve) was detected!")
else:
    print("Key exchange successful. No attacker detected (or Eve was too stealthy!).")

Generating 20 random bits and bases using Quantum Measurements...

--- Attack Details ---
Eve's Attack Pattern (1=Attacked): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Eve's Stolen Bits (None=Ignored):  [1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0]
Eve attacked 20 out of 20 qubits (100%).

Sifted Key Length: 10
Sacrificed for testing: 8 bits

Error Rate in Sample: 12.50%
Error rate exceeds 10%. An eavesdropper (Eve) was detected!
